In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, date_format, to_timestamp,col, round, when,isnan
from pyspark.sql.types import StructType,MapType, StructField, StringType
spark=SparkSession.builder.appName("Silver Layer").getOrCreate()

# Table SetUp

In [0]:
transactions=spark.table('databricks_fundamentals.bronze.transactions_data')
cards=spark.table('databricks_fundamentals.bronze.cards_data')
users=spark.table('databricks_fundamentals.bronze.users_data')

In [0]:

mcc_df = spark.read.option("multiLine", "true").json(
    "/Volumes/databricks_fundamentals/bronze/volume/mcc_codes.json"
)

mcc_dict = mcc_df.collect()[0].asDict()
mcc=spark.createDataFrame(mcc_dict.items(),['mcc_code','mcc_description'])
mcc.write.mode("overwrite").saveAsTable("databricks_fundamentals.silver.mcc_codes")

In [0]:
schema=StructType([
    StructField('target', MapType(StringType(), StringType(), True))
])
train_df = spark.read.schema(schema).json("/Volumes/databricks_fundamentals/bronze/volume/train_fraud_labels.json")

train_fraud_label=train_df.select(explode('target').alias('Id','Fraud_Label'))
train_fraud_label.write.mode('overwrite').saveAsTable('databricks_fundamentals.silver.train_fraud_labels')

# Data Cleaning

## Transactions Data

In [0]:
complete_transactions=transactions.join(mcc,transactions.mcc==mcc.mcc_code,how='inner').drop(mcc.mcc_code).join(train_fraud_label,transactions.id==train_fraud_label.Id,how='left').drop(train_fraud_label.Id)
display(complete_transactions)

id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors,mcc_description,Fraud_Label
20515493,2017-12-13T07:09:00.000Z,217,174,43.2400,Chip Transaction,31893,Aurora,IL,60503.0,5311,null,Department Stores,No
20521375,2017-12-14T10:11:00.000Z,658,199,35.0500,Online Transaction,50404,ONLINE,null,null,4784,null,Tolls and Bridge Fees,No
20515334,2017-12-13T06:32:00.000Z,490,176,23.4100,Chip Transaction,1743,Fredericksburg,TX,78624.0,5912,null,Drug Stores and Pharmacies,No
17840991,2016-05-24T07:11:00.000Z,1237,3372,1.6300,Chip Transaction,86438,Waukegan,IL,60085.0,5499,null,Miscellaneous Food Stores,No
17836278,2016-05-23T07:12:00.000Z,360,2719,1.9400,Swipe Transaction,86438,Bridgeton,NJ,8302.0,5499,null,Miscellaneous Food Stores,No
17839056,2016-05-23T15:39:00.000Z,1291,2473,63.8700,Chip Transaction,60569,Saint Louis,MO,63110.0,5300,null,Wholesale Clubs,No
17843603,2016-05-24T15:03:00.000Z,665,3652,65.0800,Chip Transaction,84008,Warren,OH,44485.0,5912,null,Drug Stores and Pharmacies,No
17835268,2016-05-22T21:01:00.000Z,264,2239,51.8600,Chip Transaction,67570,Lemoyne,PA,17043.0,5311,null,Department Stores,No
20519831,2017-12-14T02:47:00.000Z,758,5115,3.5300,Swipe Transaction,43007,Alhambra,CA,91801.0,5812,null,Eating Places and Restaurants,No
20521192,2017-12-14T09:39:00.000Z,1638,4728,8.4800,Chip Transaction,50783,South Gate,CA,90280.0,5411,null,"Grocery Stores, Supermarkets",No


In [0]:
cnt=[complete_transactions.filter(col(c).isNull()).count() for c in complete_transactions.columns]
null_df=spark.createDataFrame([cnt],complete_transactions.columns)
display(null_df)

id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors,mcc_description,Fraud_Label
0,0,0,0,0,0,0,0,1563700,1652706,0,13094522,0,4390952


In [0]:
complete_transactions.count()

13305915

In [0]:
merstateNull=complete_transactions.filter(col("merchant_state").isNull())
uniqueValues= [row[0] for row in merstateNull.select(col('merchant_city')).distinct().collect()]
nullcount=merstateNull.filter(col('zip').isNull()).count()
display(f"merstateNullCount: {merstateNull.count()}, zipNullcount:{nullcount}, Merchant cities when zip and state are null: {uniqueValues}")

"merstateNullCount: 1563700, zipNullcount:1563700, Merchant cities when zip and state are null: ['ONLINE']"

In [0]:
modified_transactions=complete_transactions.withColumn('date',to_timestamp('date','yyyy-MM-dd HH:mm:ss')).withColumn('amount', round(col("amount"),2)).withColumn('Fraud_Label',when(col('Fraud_Label')=='Yes',True).when(col('Fraud_Label')=='No',False).otherwise(None)).dropDuplicates()
modified_transactions.write.mode('overwrite').saveAsTable('databricks_fundamentals.silver.transactions')


## Cards Data

In [0]:
cnt=[cards.filter(col(c).isNull()).count() for c in cards.columns]
null_df=spark.createDataFrame([cnt],cards.columns)
display(null_df)

id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
modified_cards=cards.withColumn('credit_limit',round(col('credit_limit'),2))\
    .withColumn('has_chip', when(col('has_chip')=='YES', True).when(col('has_chip')=='NO',False).otherwise(None)).\
    withColumn('card_on_dark_web',when(col('card_on_dark_web')=='Yes',True).when(col('card_on_dark_web')=='No',False).otherwise(None)).dropDuplicates()
modified_cards.write.mode('overwrite').saveAsTable('databricks_fundamentals.silver.cards')

## Users Data

In [0]:
cnt=[users.filter(col(c).isNull()).count() for c in users.columns]
null_df=spark.createDataFrame([cnt],users.columns)
display(null_df)

id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
modified_users=users.withColumn('per_capita_income',round(col('per_capita_income'),2)).\
    withColumn('yearly_income',round(col('yearly_income'),2)).\
    withColumn('total_debt',round(col('total_debt'),2)).\
    dropDuplicates()


modified_users.write.mode('overwrite').saveAsTable('databricks_fundamentals.silver.users')


In [0]:
display(train_fraud_label)

Id,Fraud_Label
10649266,No
23410063,No
9316588,No
12478022,No
9558530,No
12532830,No
19526714,No
9906964,No
13224888,No
13749094,No


In [0]:
display(mcc)

mcc_code,mcc_description
1711,"Heating, Plumbing, Air Conditioning Contractors"
3000,Steelworks
3001,Steel Products Manufacturing
3005,Miscellaneous Metal Fabrication
3006,Miscellaneous Fabricated Metal Products
3007,Coated and Laminated Products
3008,Steel Drums and Barrels
3009,Fabricated Structural Metal Products
3058,"Tools, Parts, Supplies Manufacturing"
3066,Miscellaneous Metals
